In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [242]:
key = pd.read_csv('https://raw.githubusercontent.com/HSEAi2025Team-7/keyRateForecast/Alex/cbr.csv')

In [243]:
key.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   date             50 non-null     object 
 1   change_key_rate  50 non-null     object 
 2   basis_points     29 non-null     float64
 3   key_rate         48 non-null     float64
 4   inflation        38 non-null     float64
dtypes: float64(3), object(2)
memory usage: 2.1+ KB


In [218]:
# заполняем медианой basis_point
key['basis_points'] = key['basis_points'].fillna('0')
# заполняем медианой key_rate
med_key_rate = key['key_rate'].median()
key['key_rate'] = key['key_rate'].fillna(med_key_rate)
# заполняем пропуск медианой
med_inflation = key["inflation"].median()
key["inflation"] = key["inflation"].fillna(med_inflation)
# добавляем колонку цель по инфляции = 4
key['inflation_target'] = 4
# добавляем колонку разница между инфляцией и целью по инфляции
key['difference'] = key['inflation'] - key['inflation_target']
# добавляем колонку разница межды инфляцией
key['inflation_diff'] = key['inflation'].diff().shift(-1)
# заполняем пропуск медианой
med_inflation_diff = key["inflation_diff"].median()
key["inflation_diff"] = key["inflation_diff"].fillna(med_inflation_diff)
# прошлая ключевая ставка
key['key_rate_last'] = key['key_rate'].shift(-1)
med_key_rate_last = key['key_rate_last'].median()
key['key_rate_last'] = key['key_rate_last'].fillna(med_key_rate_last)
# на сколько изменилась ставка
key['key_rate_last_diff'] = key['key_rate_last'].diff().shift(-1)
med_key_rate_diff = key["key_rate_last_diff"].median()
key["key_rate_last_diff"] = key["key_rate_last_diff"].fillna(med_key_rate_diff)

In [219]:
key.isna().sum()

,0
date,0
change_key_rate,0
basis_points,0
key_rate,0
inflation,0
inflation_target,0
difference,0
inflation_diff,0
key_rate_last,0
key_rate_last_diff,0


In [221]:
key.head()

,date,change_key_rate,basis_points,key_rate,inflation,inflation_target,difference,inflation_diff,key_rate_last,key_rate_last_diff
0,24 октября 2025,снизить,50.0,16.5,8.2,4,4.2,0.0,17.0,1.0
1,12 сентября 2025,снизить,100.0,17.0,8.2,4,4.2,1.0,18.0,2.0
2,25 июля 2025,снизить,200.0,18.0,9.2,4,5.2,0.6,20.0,1.0
3,6 июня 2025,снизить,100.0,20.0,9.8,4,5.8,0.5,21.0,0.0
4,25 апреля 2025,сохранить,0,21.0,10.3,4,6.3,-0.1,21.0,0.0


In [222]:
# делим на матрицу признаков и целевую переменную
X1 = key[['inflation', 'inflation_target', 'difference', 'inflation_diff', 'key_rate_last', 'key_rate_last_diff']]
y1 = key['change_key_rate']

In [223]:
# делим на матрицу признаков и целевую переменную
X2 = key[['inflation', 'inflation_target', 'difference', 'inflation_diff', 'key_rate_last', 'key_rate_last_diff']]
y2 = key['key_rate']

In [224]:
X1.head()

,inflation,inflation_target,difference,inflation_diff,key_rate_last,key_rate_last_diff
0,8.2,4,4.2,0.0,17.0,1.0
1,8.2,4,4.2,1.0,18.0,2.0
2,9.2,4,5.2,0.6,20.0,1.0
3,9.8,4,5.8,0.5,21.0,0.0
4,10.3,4,6.3,-0.1,21.0,0.0


In [225]:
y1.head()

,change_key_rate
0,снизить
1,снизить
2,снизить
3,снизить
4,сохранить


Рабочая модель

In [238]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score, mean_absolute_error, mean_squared_error, r2_score, accuracy_score

# Разделение данных
X1_train, X1_test, y1_train, y1_test = train_test_split(
    X1, y1, test_size=0.20, random_state=42 , stratify=y1
)

# Масштабирование
scaler = StandardScaler()
X1_train_scaled = scaler.fit_transform(X1_train)
X1_test_scaled = scaler.transform(X1_test)

# Создание модели с фиксированными параметрами
model = LogisticRegression(
    class_weight='balanced',
    penalty= 'l2',
    solver= 'liblinear', #'saga'
)

# Обучение модели
model.fit(X1_train_scaled, y1_train)

# Оценка модели на тесте

y1_pred = model.predict(X1_test_scaled)

In [239]:
print("Accuracy:", accuracy_score(y1_test, y1_pred))
print("f1:", f1_score(y1_test, y1_pred, average='micro'))

Accuracy: 0.4
f1: 0.4


In [240]:
print("Отчёт классификации:\n", classification_report(y1_test, y1_pred,))
print("Матрица ошибок:\n", confusion_matrix(y1_test, y1_pred))

Отчёт классификации:
               precision    recall  f1-score   support

    повысить       0.50      1.00      0.67         3
     снизить       0.50      0.33      0.40         3
   сохранить       0.00      0.00      0.00         4

    accuracy                           0.40        10
   macro avg       0.33      0.44      0.36        10
weighted avg       0.30      0.40      0.32        10

Матрица ошибок:
 [[3 0 0]
 [0 1 2]
 [3 1 0]]


In [241]:
print(y1_pred)
print(y1_test)

['сохранить' 'повысить' 'повысить' 'снизить' 'повысить' 'повысить'
 'снизить' 'повысить' 'повысить' 'сохранить']
26      снизить
18     повысить
21    сохранить
29      снизить
37     повысить
43    сохранить
23    сохранить
42    сохранить
15     повысить
1       снизить
Name: change_key_rate, dtype: object


In [209]:
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score, mean_absolute_error, mean_squared_error, r2_score, accuracy_score

# your code here
# Разделение данных
X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y2, test_size=0.20, random_state=42
    )

    # Масштабирование
scaler = StandardScaler()
X2_train_scaled = scaler.fit_transform(X2_train)
X2_test_scaled = scaler.transform(X2_test)

    # Создание модели с фиксированными параметрами
model = LinearRegression()

                # Обучение модели
model.fit(X2_train_scaled, y2_train)

                # Оценка модели на тесте

y2_pred = model.predict(X2_test_scaled)

In [210]:
mse = mean_squared_error(y2_test, y2_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y2_test, y2_pred)
mape = np.mean(np.abs((y2_test - y2_pred) / y2_test)) * 100  # в процентах
R2 = r2_score(y2_test, y2_pred)
# Вывод результатов
print(f"MSE:  {mse:.6f}")
print(f"RMSE: {rmse:.6f}")
print(f"MAE:  {mae:.6f}")
print(f"MAPE: {mape:.6f}%")
print(f"R2:   {R2:.6f}")

MSE:  6.176789
RMSE: 2.485315
MAE:  1.163956
MAPE: 11.091059%
R2:   0.564267


In [211]:
print(y2_pred)
print(y2_test)

[16.22165183  5.94626706 21.61839455  4.99225001 14.52624343  4.74919567
  7.52733464  7.62853316  9.05376772  8.37568443]
13    16.00
39     5.50
30    14.00
45     4.25
17    13.00
48     5.50
26     7.50
25     7.50
32     9.00
19     8.50
Name: key_rate, dtype: float64
